In [ ]:
import os
import ray
import numpy as np
import pandas as pd
from typing import List
from utils.data import load_yaml
from imaris.imaris import ImarisDataObject
from parsers.time_step_surface_parser import TimeStepSurfaceParserDistributed
from parsers_v2.time_step_surface_parser import (
    TimeStepSurfaceParserDistributed as TimeStepSurfaceParserDistributed_v2,
)
import time

In [3]:
# ims_file = "/home/shehan/Documents/projects/nih/nih_parsers/data/temp_test/Ex112-mitotrackerred-B6-BALB-050125_batch_TileScan_1_BALB-US-Position_3_(0).ims"
ims_file = "/home/shehan/Documents/projects/nih/nih_parsers/data/tracks/GFP #1 Sec2 Roi1 2x2 1h30min.ims"
parser = TimeStepSurfaceParserDistributed(ims_file_path=ims_file)
parser_v2 = TimeStepSurfaceParserDistributed_v2(ims_file_path=ims_file)
ims = ImarisDataObject(ims_file)
data = ims.data

In [6]:
start = time.perf_counter()
out = parser.inspect(0)
stop = time.perf_counter()
print(f"run time: {stop - start}")

run time: 6.4666946299839765


In [7]:
start = time.perf_counter()
out_v2 = parser_v2.inspect(0)
stop = time.perf_counter()
print(f"run time: {stop - start}")

run time: 0.7629606011323631


In [11]:
final = out["final_df"]
final_v2 = out_v2["final_df"]
print(final.equals(final_v2))
print(final_v2.equals(final))

True
True


In [6]:
f = out["stat_names_channel_info"]
f2 = out["stat_names_channel_info_fast"]
f.equals(f2)

True

In [7]:
f = out["stat_names_surface_info"]
f2 = out["stat_names_surface_info_fast"]
f.equals(f2)

True

In [5]:
f = out["stats_df"]
f2 = out["stats_df2"]
f.equals(f2)

True

In [ ]:
f2

In [ ]:
temp = []
for i in range(len(f)):
    if not f.iloc[i].equals(f2.iloc[i]):
        # print(f.iloc[i])
        # print(f2.iloc[i])
        # print("\n")
        temp.append(i)

len(temp)
temp

In [ ]:
out["stat_names_raw"]

In [ ]:
f.iloc[8081]

In [ ]:
f2.iloc[8081]

In [ ]:
out["factor"]

In [ ]:
out["final_df"]["Object_ID"].unique().shape

In [ ]:
s = ims.get_stats_names("MegaSurfaces0")
s[s["Name"] == "Time"]

In [ ]:
ims.get_object_factor("MegaSurfaces0")

In [ ]:
stats_values = out["stat_values_raw"]
stats_names = out["stat_names_raw"]
time_index_id = stats_names[stats_names["Name"] == "Time Index"]["ID"]
time_index_id = time_index_id.iloc[0].item()
time_index_id

In [ ]:
stats_values[stats_values["ID_StatisticsType"] == 26179]

In [ ]:
filter_col_names = ["ID_Object", "ID_StatisticsType", "Value"]
object_id = out["object_id"]
stats_values = ims.get_stats_values("MegaSurfaces0")
filter_values = [
    object_id,
    pd.Series([time_index_id]),
    # pd.Series([1.0]),
]

num_objects = 0

for i in range(80):
    stats_values = ims.get_stats_values("MegaSurfaces0")
    filter_values = [
        object_id,
        pd.Series([time_index_id]),
        pd.Series([i]),
    ]

    for col_names, values in zip(filter_col_names, filter_values):
        # print(type(col_names), type(values))
        stats_values = stats_values[stats_values[col_names].isin(values)]

    num_objects += len(stats_values)

In [ ]:
num_objects

In [ ]:
out["object_id"]

In [ ]:
stats_values = ims.get_stats_values("MegaSurfaces0")
stats_values = stats_values[stats_values["ID_Object"].isin(stats_values["ID_Object"])]

In [ ]:
stats_values

In [ ]:
grouped_stats = (
    stats_values.groupby("ID_Object")[["ID_StatisticsType", "Value"]]
    .apply(lambda x: x.set_index("ID_StatisticsType").to_dict(orient="dict"))
    .to_dict()
)
grouped_stats = {k: v["Value"] for k, v in grouped_stats.items()}

In [ ]:
f = out["final_df"]
f2 = out["final_df2"]
f.equals(f2)

In [ ]:
f

In [ ]:
np.array(data.get("Scene8").get("Content").get("MegaSurfaces0").get("SurfaceModel"))

In [ ]:
np.array(data.get("Scene8").get("Content").get("MegaSurfaces0").get("TrackObject0"))

In [ ]:
object_ids = pd.DataFrame(
    np.asarray(
        data.get("Scene8").get("Content").get("MegaSurfaces0").get("TrackObject0")
    )
)
object_ids

In [ ]:
stats_id_objects = pd.DataFrame(
    np.asarray(
        data.get("Scene8").get("Content").get("MegaSurfaces0").get("StatisticsValue")
    )
)
stats_id_objects

In [ ]:
np.unique(stats_id_objects["ID_Object"]).shape

In [ ]:
np.unique(object_ids).shape

In [ ]:
stats_id_objects["ID_Object"]

In [ ]:
object_ids = pd.DataFrame(
    np.asarray(
        data.get("Scene8").get("Content").get("MegaSurfaces0").get("StatisticsValue")
    )
)
object_ids

In [ ]:
trackinfo = pd.DataFrame(
    np.asarray(data.get("Scene8").get("Content").get("MegaSurfaces0").get("Track0"))
)
trackinfo

In [ ]:
track_ids = pd.DataFrame(
    np.asarray(data.get("Scene8").get("Content").get("MegaSurfaces0").get("Track0"))
)["ID"]

In [ ]:
track_ids

In [ ]:
obids = stats_id_objects["ID_Object"]
obids

In [ ]:
stats_id_objects[~stats_id_objects["ID_Object"].isin(track_ids)]

In [ ]:
TrackSegment0

In [ ]:
out = parser.inspect(0)

In [ ]:
f = out["stats_df"]
f2 = out["stats_df2"]
f.equals(f2)